# What a convolution actually buys you

MichAl Academy, lesson 3.4.

Run each cell with **Shift+Enter**.

A convolution is usually sold as translation invariance. This notebook trains
two architectures, moves the test digits by one pixel, and finds out.


In [ ]:
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.set_num_threads(1)
np.set_printoptions(precision=3, suppress=True)

digits = load_digits()
X = digits.images.astype(np.float32) / 16.0     # (N, 8, 8)
y = digits.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

print(f"images: {X.shape}, train {len(Xtr)}, test {len(Xte)}")


Moving an image is the whole experiment, so define it plainly: shift the pixels
by whole positions and fill the vacated edge with zero.


In [ ]:
def shift(imgs, dx, dy):
    out = np.zeros_like(imgs)
    h, w = imgs.shape[1], imgs.shape[2]
    ys0, ys1 = max(0, dy), min(h, h + dy)
    xs0, xs1 = max(0, dx), min(w, w + dx)
    out[:, ys0:ys1, xs0:xs1] = imgs[:, ys0 - dy:ys1 - dy, xs0 - dx:xs1 - dx]
    return out


example = Xtr[0]
print("original:")
print(np.round(example * 16).astype(int))
print("\nmoved one pixel right:")
print(np.round(shift(Xtr[:1], 1, 0)[0] * 16).astype(int))


In [ ]:
def tens(imgs, conv):
    t = torch.tensor(imgs)
    return t.unsqueeze(1) if conv else t.reshape(len(imgs), -1)


def make_mlp(seed):
    torch.manual_seed(seed)
    return torch.nn.Sequential(
        torch.nn.Linear(64, 32), torch.nn.ReLU(), torch.nn.Linear(32, 10))


def make_cnn(seed):
    torch.manual_seed(seed)
    return torch.nn.Sequential(
        torch.nn.Conv2d(1, 8, 3, padding=1), torch.nn.ReLU(),
        torch.nn.MaxPool2d(2),
        torch.nn.Conv2d(8, 16, 3, padding=1), torch.nn.ReLU(),
        torch.nn.MaxPool2d(2),
        torch.nn.Flatten(), torch.nn.Linear(16 * 2 * 2, 10))


def train(make, conv, seed, imgs, labels, epochs=30, batch=64, lr=1e-3):
    net = make(seed)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    lossf = torch.nn.CrossEntropyLoss()
    Xt, yt = tens(imgs, conv), torch.tensor(labels)
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(Xt), batch):
            idx = perm[i:i + batch]
            opt.zero_grad()
            lossf(net(Xt[idx]), yt[idx]).backward()
            opt.step()
    return net


def acc(net, conv, imgs, labels):
    with torch.no_grad():
        return (net(tens(imgs, conv)).argmax(1) == torch.tensor(labels)).float().mean().item()


## 1. Parameter counts before anything is trained


In [ ]:
for name, make in (("dense", make_mlp), ("convolutional", make_cnn)):
    print(f"{name:<15}{sum(p.numel() for p in make(0).parameters()):>6} parameters")


## 2. Train both on centred digits


In [ ]:
SEEDS = 5
dense_nets = [train(make_mlp, False, s, Xtr, ytr) for s in range(SEEDS)]
cnn_nets = [train(make_cnn, True, s, Xtr, ytr) for s in range(SEEDS)]

print(f"dense         {np.median([acc(n, False, Xte, yte) for n in dense_nets]):.4f}")
print(f"convolutional {np.median([acc(n, True, Xte, yte) for n in cnn_nets]):.4f}")


Half a point apart, with the convolutional network using a fifth fewer
parameters. On centred, pre-cropped, well-behaved digits that is the whole
benefit.

## 3. Now move the digit


In [ ]:
SHIFTS = [(0, 0), (1, 0), (0, 1), (1, 1), (2, 0)]

print(f"{'shift':<10}{'dense':>10}{'conv':>10}")
for dx, dy in SHIFTS:
    Xs = shift(Xte, dx, dy)
    d = np.median([acc(n, False, Xs, yte) for n in dense_nets])
    c = np.median([acc(n, True, Xs, yte) for n in cnn_nets])
    label = "none" if (dx, dy) == (0, 0) else f"{dx:+d},{dy:+d}"
    print(f"{label:<10}{d:>10.4f}{c:>10.4f}")


Both collapse.

This is worth being blunt about. **A convolution does not give you translation
invariance.** Sliding one set of weights everywhere means a feature detector
fires wherever its feature is, so the feature map moves when the input moves.
That is equivariance. The classifier sitting on top still reads a fixed grid of
positions, and it has only ever seen centred digits.

Pooling buys a little tolerance, and a little is what the gap between the two
columns shows.

## 4. So what does fix it?

Show the networks a shifted digit during training.


In [ ]:
aug_imgs = np.concatenate([Xtr] + [shift(Xtr, dx, dy)
                                   for dx, dy in ((1, 0), (0, 1), (-1, 0), (0, -1))])
aug_y = np.concatenate([ytr] * 5)
print(f"augmented training set: {len(aug_imgs)} images")

aug_dense = [train(make_mlp, False, s, aug_imgs, aug_y) for s in range(SEEDS)]
aug_cnn = [train(make_cnn, True, s, aug_imgs, aug_y) for s in range(SEEDS)]

print(f"\n{'shift':<10}{'dense':>10}{'conv':>10}{'dense+aug':>12}{'conv+aug':>12}")
for dx, dy in SHIFTS:
    Xs = shift(Xte, dx, dy)
    row = (
        np.median([acc(n, False, Xs, yte) for n in dense_nets]),
        np.median([acc(n, True, Xs, yte) for n in cnn_nets]),
        np.median([acc(n, False, Xs, yte) for n in aug_dense]),
        np.median([acc(n, True, Xs, yte) for n in aug_cnn]),
    )
    label = "none" if (dx, dy) == (0, 0) else f"{dx:+d},{dy:+d}"
    print(f"{label:<10}{row[0]:>10.4f}{row[1]:>10.4f}{row[2]:>12.4f}{row[3]:>12.4f}")


Read that table twice, because it has two halves and they matter equally.

**The data did the heavy lifting.** The dense network went from about 0.41 to
about 0.90 on the one-pixel shift with no architecture change at all.

**The architecture decided what the data was worth.** Given the identical
augmented training set the convolutional network is far ahead, and further ahead
the harder the shift gets, while using fewer parameters.

So: a convolution is not a guarantee of anything. It is a prior that makes
translation cheap to learn, and it only pays out if the training data contains
the variation it was built to exploit.

## 5. What the first layer decided to look for


In [ ]:
w = cnn_nets[0][0].weight.detach().numpy()
print(f"first conv layer: {w.shape[0]} filters of {w.shape[2]}x{w.shape[3]}\n")
for i in range(3):
    print(f"filter {i}:\n{np.round(w[i, 0], 3)}\n")


## 6. What you have

- On an easy centred task a convolution buys parameter efficiency, not accuracy.
- Convolution gives equivariance, not invariance, and one pixel is enough to
  prove it.
- Augmentation supplies the variation; the architecture decides how efficiently
  that variation is converted into robustness.
- A model that scores 0.94 centred and 0.11 two pixels over has a property an
  attacker can use, which is where lesson 7.4 picks up.

Lesson 3.5 removes the labels entirely.
